#Configuration

In [0]:
CATALOG = "cms_medicare_databricks_pipeline"
SOURCE_PROCEDURES = f"{CATALOG}.silver.procedures"
SOURCE_PROVIDERS  = f"{CATALOG}.silver.providers"
TARGET_COST       = f"{CATALOG}.gold.provider_cost_scorecard"

#Temp Views

In [0]:
spark.sql(f"CREATE OR REPLACE TEMP VIEW v_procedures AS SELECT * FROM {SOURCE_PROCEDURES}")
spark.sql(f"CREATE OR REPLACE TEMP VIEW v_providers AS SELECT * FROM {SOURCE_PROVIDERS}")

print("Temp views created")
print(f"  v_procedures -> {SOURCE_PROCEDURES}")
print(f"  v_providers  -> {SOURCE_PROVIDERS}")

#Gold Table: Provider Cost Scorecard

In [0]:
%sql
--Business Question: Which California providers have the highest cost variation for common procedures vs the state average?
CREATE OR REPLACE TABLE cms_medicare_databricks_pipeline.gold.provider_cost_scorecard
-- Liquid clustering for query performance
CLUSTER BY (hcpcs_code, region, cost_outlier)
AS

WITH procedure_benchmarks AS (
    SELECT
        hcpcs_code,
        hcpcs_description,
        place_of_service,
        COUNT(DISTINCT provider_npi) AS total_providers_billing,
        ROUND(AVG(avg_medicare_standardized_amount), 2) AS state_avg_payment,
        ROUND(STDDEV(avg_medicare_standardized_amount), 2) AS state_stddev_payment,
        ROUND(PERCENTILE(avg_medicare_standardized_amount, 0.25), 2) AS p25_payment,
        ROUND(PERCENTILE(avg_medicare_standardized_amount, 0.50), 2) AS median_payment,
        ROUND(PERCENTILE(avg_medicare_standardized_amount, 0.75), 2) AS p75_payment
    FROM v_procedures
    WHERE avg_medicare_standardized_amount IS NOT NULL
    GROUP BY all
),

provider_with_benchmarks AS (
    SELECT
        -- Provider Identity
        p.provider_npi,
        prov.provider_last_name,
        prov.provider_first_name,
        prov.credentials,
        prov.provider_type,
        prov.city,
        prov.zip_code,
        prov.ruca_code,
        -- Procedure identity
        p.hcpcs_code,
        p.hcpcs_description,
        p.place_of_service,
        -- Volume
        p.total_beneficiaries,
        p.total_services,
        -- Cost columns
        p.avg_submitted_charge,
        p.avg_medicare_allowed_amount,
        p.avg_medicare_payment_amount,
        p.avg_medicare_standardized_amount,
        -- State benchmarks for this procedure
        b.state_avg_payment,
        b.state_stddev_payment,
        b.median_payment,
        b.p25_payment,
        b.p75_payment,
        b.total_providers_billing,
        -- Dollar deviation from state average
        -- Positive = cost more than average, Negative = costs less than average
        ROUND(p.avg_medicare_standardized_amount - b.state_avg_payment, 2) AS deviation_from_state_avg,
        -- Percentage deviation from state average
        ROUND(((p.avg_medicare_standardized_amount - b.state_avg_payment) / NULLIF(b.state_avg_payment, 0)) * 100, 2) AS pct_deviation_from_state_avg,
        -- Flag cost outliers
        CASE
            WHEN p.avg_medicare_standardized_amount > b.p75_payment THEN 'High'
            WHEN p.avg_medicare_standardized_amount < b.p25_payment THEN 'Low'
            ELSE 'Normal'
        END AS cost_outlier,
        -- Orange County Flag
        CASE
            WHEN prov.zip_code LIKE '926%' 
                OR prov.zip_code LIKE '927%'
                OR prov.zip_code LIKE '928%' 
            THEN 'orange_county'
            ELSE 'other_region'
        END AS region,
        -- Billing ratio: what they submitted vs what Medicare paid
        ROUND(p.avg_submitted_charge / NULLIF(p.avg_medicare_payment_amount, 0),2) AS billing_ratio
    FROM v_procedures p
    INNER JOIN v_providers prov
        ON p.provider_npi = prov.provider_npi
    INNER JOIN procedure_benchmarks b
        ON p.hcpcs_code = b.hcpcs_code
        AND p.place_of_service = b.place_of_service
    WHERE p.avg_medicare_standardized_amount IS NOT NULL
)

-- Final output: all columns ordered by deviation descending
SELECT *
FROM provider_with_benchmarks
ORDER BY pct_deviation_from_state_avg DESC

#Verify

In [0]:
%sql
-- Row counts by cost outlier category
SELECT
    cost_outlier,
    region,
    COUNT(*) AS row_count,
    ROUND(AVG(pct_deviation_from_state_avg), 2) AS avg_pct_deviation,
    ROUND(AVG(billing_ratio), 2)          AS avg_billing_ratio
FROM cms_medicare_databricks_pipeline.gold.provider_cost_scorecard
GROUP BY cost_outlier, region
ORDER BY cost_outlier, region

### OC preview

In [0]:
%sql
-- Top 10 high cost Orange County providers
SELECT
    provider_last_name,
    provider_first_name,
    credentials,
    city,
    hcpcs_description,
    avg_medicare_standardized_amount,
    state_avg_payment,
    pct_deviation_from_state_avg,
    billing_ratio,
    cost_outlier
FROM cms_medicare_databricks_pipeline.gold.provider_cost_scorecard
WHERE region = 'orange_county'
  AND cost_outlier = 'High'
ORDER BY pct_deviation_from_state_avg DESC
LIMIT 10

#Optimize

In [0]:
%sql
-- Optimize for query performance and compaction
OPTIMIZE cms_medicare_databricks_pipeline.gold.provider_cost_scorecard